In [61]:
import sys
!"{sys.executable}" -m pip install opencv-python


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
import cv2

print(cv2.__version__)

4.10.0


In [63]:
import cv2
import numpy as np 
from matplotlib import pyplot as plt
%matplotlib inline 

# Import required modules
import sys
# Uninstall the broken OpenCV 5 beta versions
!{sys.executable} -m pip uninstall -y cv2 opencv-python opencv-contrib-python
# Force install the latest stable OpenCV 4.10
!{sys.executable} -m pip install opencv-python==4.10.0.84 opencv-contrib-python==4.10.0.84

Found existing installation: opencv-python 4.10.0.84
Uninstalling opencv-python-4.10.0.84:
  Successfully uninstalled opencv-python-4.10.0.84
Found existing installation: opencv-contrib-python 4.10.0.84
Uninstalling opencv-contrib-python-4.10.0.84:
  Successfully uninstalled opencv-contrib-python-4.10.0.84


You can safely remove it manually.


  Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)
Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl (45.5 MB)

   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   ---------------------------------------- 2/2 [opencv-contrib-python]




[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
# train HOG + SVM

In [65]:
from pathlib import Path, PureWindowsPath
from collections import Counter
import csv
import json
import pickle

import cv2
import numpy as np

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

# Use the same six-class dataset root as step1.
DATASET_ROOT = Path(
    r"C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project\mouth_viseme_dataset"
)

# Keep the original person IDs and add new videos here.
PERSON_BY_VIDEO = {
    "train01_asian_female_12s.mp4": "p01",
    "train02_middleeastern_female_hijab_10s.mp4": "p02",
    "Ishihara Satomi.mp4": "p03",
    "train04_white_female_10s.mp4": "p04",
    "Elon Musk.mp4": "p05",
    "train06a_asian_male_6s.mp4": "p06",
    "Rachels English.mp4": "p07",
    "train08_black_male_suit_10s.mp4": "p08",
    "San Diego Voice and Accent.mp4": "p09",
    "Michelle C. Smith.mp4": "p10",
    "8724461-hd_720_1366_25fps.mp4": "p11"
}

TRAIN_PEOPLE = {
    "p01", "p02", "p03", "p04", "p05",
    "p06", "p07", "p09", "p11"
}
VAL_PEOPLE = {"p10"}
TEST_PEOPLE = {"p08"}

LABEL_NAMES = ["closed", "a", "i", "u", "e", "o"]

LABEL_TO_ID = {
    name: index
    for index, name in enumerate(LABEL_NAMES)
}
LABEL_IDS = list(range(len(LABEL_NAMES)))

# Store the semantic mapping for later VRM animation.
VRM_EXPRESSION_BY_LABEL = {
    "closed": None,
    "a": "aa",
    "i": "ih",
    "u": "ou",
    "e": "ee",
    "o": "oh"
}

records = []
seen = {}
duplicate_count = 0

csv_paths = sorted(
    DATASET_ROOT.rglob("labels.csv")
)

if not csv_paths:
    raise FileNotFoundError(
        f"No six-class label files found: {DATASET_ROOT}"
    )

# CSV → validation → deduplication → sample records.
for csv_path in csv_paths:
    with open(
        csv_path,
        newline="",
        encoding="utf-8-sig"
    ) as file:

        for row in csv.DictReader(file):
            label = row["label"].strip().lower()

            if label not in LABEL_TO_ID:
                raise ValueError(
                    f"Invalid mouth-shape label: {label}, source: {csv_path}"
                )

            frame_index = int(row["frame_index"])

            if frame_index < 0:
                raise ValueError(
                    f"Found a negative frame index. Check for overwritten images: {csv_path}"
                )

            if row["mouth"].strip() != "mouth":
                raise ValueError(
                    f"Found a sample that is not a mouth crop: {csv_path}"
                )

            video_name = PureWindowsPath(
                row["source_video"].replace("/", "\\")
            ).name

            key = (
                video_name,
                frame_index,
                "mouth"
            )

            image_path = csv_path.parent / row["filename"]

            if not image_path.is_file():
                raise FileNotFoundError(image_path)

            if key in seen:
                if seen[key] != label:
                    raise ValueError(
                        "Conflicting labels for the same video frame. Fix them manually: "
                        f"{key}"
                    )

                duplicate_count += 1
                continue

            seen[key] = label

            records.append({
                "image_path": image_path,
                "video": video_name,
                "frame_index": frame_index,
                "mouth": "mouth",
                "label": label,
                "y": LABEL_TO_ID[label]
            })

video_names = sorted({
    record["video"]
    for record in records
})

print("Source videos and sample counts:")

for name in video_names:
    counts = Counter(
        record["label"]
        for record in records
        if record["video"] == name
    )

    print(
        name,
        " | ",
        ", ".join(
            f"{label}={counts[label]}"
            for label in LABEL_NAMES
        )
    )

print("Duplicate samples ignored:", duplicate_count)

missing_names = [
    name for name in video_names
    if name not in PERSON_BY_VIDEO
]

if missing_names:
    raise ValueError(
        "Add person IDs for these videos in PERSON_BY_VIDEO: "
        f"{missing_names}"
    )

for record in records:
    record["person"] = PERSON_BY_VIDEO[
        record["video"]
    ]

# Keep each person in exactly one split.
if (
    TRAIN_PEOPLE & VAL_PEOPLE
    or TRAIN_PEOPLE & TEST_PEOPLE
    or VAL_PEOPLE & TEST_PEOPLE
):
    raise ValueError(
        "Training, validation and test people must not overlap."
    )

expected_people = (
    TRAIN_PEOPLE | VAL_PEOPLE | TEST_PEOPLE
)
actual_people = {
    record["person"]
    for record in records
}

if actual_people != expected_people:
    raise ValueError(
        f"Person IDs do not match. Found: {actual_people}. "
        f"Expected for the split: {expected_people}"
    )

for split_name, people in [
    ("train", TRAIN_PEOPLE),
    ("validation", VAL_PEOPLE),
    ("test", TEST_PEOPLE)
]:
    selected = [
        record for record in records
        if record["person"] in people
    ]

    counts = Counter(
        record["label"]
        for record in selected
    )

    print(
        f"{split_name}: people={sorted(people)} | "
        + ", ".join(
            f"{label}={counts[label]}"
            for label in LABEL_NAMES
        )
    )

    missing = [
        label for label in LABEL_NAMES
        if counts[label] == 0
    ]

    # Require all six classes for training and validation.
    if missing and split_name in (
        "train", "validation"
    ):
        raise ValueError(
            f"{split_name} is missing these mouth shapes: {missing}. "
            "Add labeled samples."
        )

    # Explicitly report classes absent from the test set.
    if missing:
        print(
            f"Note: The test set has no samples for {missing}. "
            "Recall cannot be measured for these classes."
        )

Source videos and sample counts:
8724461-hd_720_1366_25fps.mp4  |  closed=68, a=79, i=69, u=121, e=102, o=45
Elon Musk.mp4  |  closed=166, a=0, i=28, u=1, e=8, o=25
Ishihara Satomi.mp4  |  closed=78, a=0, i=111, u=10, e=20, o=0
Michelle C. Smith.mp4  |  closed=55, a=28, i=157, u=101, e=52, o=38
Rachels English.mp4  |  closed=168, a=5, i=182, u=73, e=79, o=58
San Diego Voice and Accent.mp4  |  closed=60, a=58, i=140, u=77, e=102, o=13
train01_asian_female_12s.mp4  |  closed=31, a=15, i=131, u=76, e=27, o=0
train02_middleeastern_female_hijab_10s.mp4  |  closed=23, a=33, i=132, u=13, e=91, o=0
train04_white_female_10s.mp4  |  closed=75, a=0, i=117, u=16, e=35, o=0
train06a_asian_male_6s.mp4  |  closed=7, a=0, i=67, u=20, e=17, o=0
train08_black_male_suit_10s.mp4  |  closed=17, a=3, i=140, u=9, e=66, o=0
Duplicate samples ignored: 0
train: people=['p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p09', 'p11'] | closed=676, a=190, i=977, u=407, e=481, o=141
validation: people=['p10'] | clos

In [66]:
# OpenCV size order is (width, height)
MOUTH_SIZE = (64, 32)

HOG_PARAMS = {
    "_winSize": MOUTH_SIZE,
    "_blockSize": (16, 16),
    "_blockStride": (8, 8),
    "_cellSize": (8, 8),
    "_nbins": 9
}

hog = cv2.HOGDescriptor(**HOG_PARAMS)


def extract_mouth_hog(image_path):
    """
    Read a grayscale mouth crop, resize it and extract a HOG vector.
    Read a grayscale mouth crop → resize → extract HOG.

    Input: path to the original mouth crop.
    Input: path to a raw mouth crop.

    Output: a one-dimensional feature vector of fixed length.
    Output: a fixed-length feature vector.
    """

    image_bytes = np.fromfile(
        str(image_path),
        dtype=np.uint8
    )

    image = cv2.imdecode(
        image_bytes,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise RuntimeError(
            f"Cannot read image: {image_path}"
        )

    interpolation = (
        cv2.INTER_AREA
        if image.shape[1] >= MOUTH_SIZE[0]
        and image.shape[0] >= MOUTH_SIZE[1]
        else cv2.INTER_LINEAR
    )

    resized = cv2.resize(
        image,
        MOUTH_SIZE,
        interpolation=interpolation
    )

    return hog.compute(resized).ravel()


def build_features(selected_records):
    """
    Convert the sample list into feature matrix X and label array y.
    Convert sample records into feature matrix X and labels y.

    Each row of X contains HOG features for one mouth crop.
    Each row of X contains one image's HOG features.

    Each entry in y contains the manual label for the same image.
    Each entry of y contains the corresponding manual class.
    """

    if not selected_records:
        raise ValueError("This split has no samples.")

    X = np.stack([
        extract_mouth_hog(record["image_path"])
        for record in selected_records
    ]).astype(np.float32)

    y = np.array([
        record["y"]
        for record in selected_records
    ], dtype=np.int32)

    return X, y


train_records = [
    record for record in records
    if record["person"] in TRAIN_PEOPLE
]

val_records = [
    record for record in records
    if record["person"] in VAL_PEOPLE
]

test_records = [
    record for record in records
    if record["person"] in TEST_PEOPLE
]

X_train, y_train = build_features(
    train_records
)
X_val, y_val = build_features(
    val_records
)

print("Training feature matrix:", X_train.shape)
print("Validation feature matrix:", X_val.shape)

best_model = None
best_params = None
best_score = -1.0
validation_results = []

# Keep the original grid; SVC handles the six-class targets.
for kernel in ("linear", "rbf"):
    for C in (0.1, 1.0, 10.0):
        for class_weight in (None, "balanced"):
            params = {
                "kernel": kernel,
                "C": C,
                "gamma": "scale",
                "class_weight": class_weight
            }

            model = SVC(**params)
            model.fit(X_train, y_train)

            prediction = model.predict(X_val)

            precision, recall, f1, support = (
                precision_recall_fscore_support(
                    y_val,
                    prediction,
                    labels=LABEL_IDS,
                    zero_division=0
                )
            )

            # Average F1 across all six classes.
            macro_f1 = float(np.mean(f1))

            result = {
                **params,
                "val_accuracy": float(
                    accuracy_score(
                        y_val, prediction
                    )
                ),
                "val_macro_f1": macro_f1
            }

            for index, label in enumerate(LABEL_NAMES):
                result[
                    f"val_{label}_precision"
                ] = float(precision[index])

                result[
                    f"val_{label}_recall"
                ] = float(recall[index])

                result[
                    f"val_{label}_f1"
                ] = float(f1[index])

            validation_results.append(result)

            print(
                f"kernel={kernel:6s}, C={C:4.1f}, "
                f"weight={str(class_weight):8s} | "
                f"macro-F1={macro_f1:.3f}, "
                f"accuracy={result['val_accuracy']:.3f}"
            )

            # Tune only on validation data; keep the first tie.
            if macro_f1 > best_score:
                best_score = macro_f1
                best_params = params.copy()
                best_model = model

print("\nBest parameters:", best_params)
print("Best validation macro-F1:", best_score)

print("\nValidation results for the best configuration:")

print(classification_report(
    y_val,
    best_model.predict(X_val),
    labels=LABEL_IDS,
    target_names=LABEL_NAMES,
    digits=4,
    zero_division=0
))

print(
    "Validation confusion matrix. Class order:",
    LABEL_NAMES
)

print(confusion_matrix(
    y_val,
    best_model.predict(X_val),
    labels=LABEL_IDS
))

Training feature matrix: (2872, 756)
Validation feature matrix: (431, 756)
kernel=linear, C= 0.1, weight=None     | macro-F1=0.163, accuracy=0.343
kernel=linear, C= 0.1, weight=balanced | macro-F1=0.222, accuracy=0.350
kernel=linear, C= 1.0, weight=None     | macro-F1=0.205, accuracy=0.355
kernel=linear, C= 1.0, weight=balanced | macro-F1=0.215, accuracy=0.357
kernel=linear, C=10.0, weight=None     | macro-F1=0.211, accuracy=0.346
kernel=linear, C=10.0, weight=balanced | macro-F1=0.204, accuracy=0.350
kernel=rbf   , C= 0.1, weight=None     | macro-F1=0.142, accuracy=0.325
kernel=rbf   , C= 0.1, weight=balanced | macro-F1=0.134, accuracy=0.251
kernel=rbf   , C= 1.0, weight=None     | macro-F1=0.156, accuracy=0.353
kernel=rbf   , C= 1.0, weight=balanced | macro-F1=0.190, accuracy=0.360
kernel=rbf   , C=10.0, weight=None     | macro-F1=0.184, accuracy=0.355
kernel=rbf   , C=10.0, weight=balanced | macro-F1=0.197, accuracy=0.362

Best parameters: {'kernel': 'linear', 'C': 0.1, 'gamma': 'sc

In [67]:
# Use the selected parameters and combine training and validation data for the final fit
X_fit = np.concatenate([X_train, X_val], axis=0)
y_fit = np.concatenate([y_train, y_val], axis=0)

final_model = SVC(**best_params)
final_model.fit(X_fit, y_fit)

# Evaluate the held-out test set after parameter selection.
X_test, y_test = build_features(test_records)
test_prediction = final_model.predict(X_test)

test_report = classification_report(
    y_test,
    test_prediction,
    labels=LABEL_IDS,
    target_names=LABEL_NAMES,
    digits=4,
    zero_division=0
)

test_report_dict = classification_report(
    y_test,
    test_prediction,
    labels=LABEL_IDS,
    target_names=LABEL_NAMES,
    output_dict=True,
    zero_division=0
)

test_matrix = confusion_matrix(
    y_test,
    test_prediction,
    labels=LABEL_IDS
)

# Select the constant-prediction baseline from training data.
baseline_id = int(
    np.bincount(
        y_train,
        minlength=len(LABEL_NAMES)
    ).argmax()
)

baseline_prediction = np.full_like(
    y_test, baseline_id
)

baseline_accuracy = accuracy_score(
    y_test, baseline_prediction
)

print("Final test people:", sorted(TEST_PEOPLE))

print(
    f"\nPredict-all-{LABEL_NAMES[baseline_id]} "
    f"baseline accuracy: {baseline_accuracy:.4f}"
)

print("\nSix-class mouth-shape HOG + SVM test results:")
print(test_report)

print("Confusion matrix: rows are true classes, columns are predicted classes")
print("Class order:", LABEL_NAMES)
print(test_matrix)

# Save each experiment in a separate run directory.
from datetime import datetime
import uuid

run_name = (
    datetime.now().strftime("%Y%m%d_%H%M%S")
    + "_"
    + uuid.uuid4().hex[:6]
)

RUN_DIR = (
    DATASET_ROOT.parent
    / "mouth_viseme_training_runs"
    / run_name
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

# Bundle the model, preprocessing and label semantics.
model_bundle = {
    "task": "mouth_viseme_6_v1",
    "model": final_model,
    "hog_params": HOG_PARAMS,
    "mouth_size": MOUTH_SIZE,
    "label_to_id": LABEL_TO_ID,
    "vrm_expression_by_label": VRM_EXPRESSION_BY_LABEL,
    "best_params": best_params,
    "person_by_video": PERSON_BY_VIDEO,
    "training_people": sorted(
        TRAIN_PEOPLE | VAL_PEOPLE
    ),
    "test_people": sorted(TEST_PEOPLE),
    "preprocessing": (
        "grayscale, resize only; no equalization"
    )
}

with open(
    RUN_DIR / "mouth_viseme_hog_svm.pkl",
    "wb"
) as file:
    pickle.dump(model_bundle, file)

with open(
    RUN_DIR / "validation_results.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(
            validation_results[0].keys()
        )
    )
    writer.writeheader()
    writer.writerows(validation_results)

# Save the actual sample split for reproducibility.
with open(
    RUN_DIR / "sample_manifest.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:

    fieldnames = [
        "image_path", "video", "frame_index",
        "mouth", "person", "label", "split"
    ]

    writer = csv.DictWriter(
        file, fieldnames=fieldnames
    )
    writer.writeheader()

    for record in records:
        split = (
            "train"
            if record["person"] in TRAIN_PEOPLE
            else "validation"
            if record["person"] in VAL_PEOPLE
            else "test"
        )

        writer.writerow({
            **{
                key: str(record[key])
                for key in fieldnames
                if key != "split"
            },
            "split": split
        })

with open(
    RUN_DIR / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "label_order": LABEL_NAMES,
            "best_params": best_params,
            "test_report": test_report_dict,
            "confusion_matrix": test_matrix.tolist(),
            "baseline_class": LABEL_NAMES[baseline_id],
            "baseline_accuracy": float(
                baseline_accuracy
            )
        },
        file,
        ensure_ascii=False,
        indent=2
    )

with open(
    RUN_DIR / "test_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(test_report)

    file.write(
        "\nClass order:" + str(LABEL_NAMES) + "\n"
    )
    file.write(str(test_matrix))

    file.write(
        f"\nBaseline class: {LABEL_NAMES[baseline_id]}. "
        f"Accuracy: {baseline_accuracy:.6f}\n"
    )

print("\nSix-class model and experiment records saved to:")
print(RUN_DIR)

Final test people: ['p08']

Predict-all-i baseline accuracy: 0.5957

Six-class mouth-shape HOG + SVM test results:
              precision    recall  f1-score   support

      closed     0.0000    0.0000    0.0000        17
           a     0.0000    0.0000    0.0000         3
           i     0.8721    0.5357    0.6637       140
           u     0.0604    1.0000    0.1139         9
           e     0.0000    0.0000    0.0000        66
           o     0.0000    0.0000    0.0000         0

    accuracy                         0.3574       235
   macro avg     0.1554    0.2560    0.1296       235
weighted avg     0.5219    0.3574    0.3998       235

Confusion matrix: rows are true classes, columns are predicted classes
Class order: ['closed', 'a', 'i', 'u', 'e', 'o']
[[ 0  0  0 17  0  0]
 [ 0  0  0  3  0  0]
 [ 0  0 75 65  0  0]
 [ 0  0  0  9  0  0]
 [ 0  0 11 55  0  0]
 [ 0  0  0  0  0  0]]

Six-class model and experiment records saved to:
C:\Users\user\Desktop\M2\Computer Vision and 